# Train the ball detector for broadcast and gym footage (WASB fine-tuning)

Change `train-ball-detector`. Builds training clips from the Volleyball dataset (broadcast) and the volleyball-orel.ru
gym videos (`scripts/build_ball_clips.py`), fine-tunes WASB's HRNet from its Volleyball-dataset weights
(`training/ball/train_wasb.py`) in two mixes, and evaluates every model on the held-out broadcast matches and the
held-out gym matches with `scripts/eval_ball_detection.py`:

| run | data |
|---|---|
| `gym` | orel only |
| `mixed` | Volleyball dataset 0.5 + orel 0.5 |

Runs on **Kaggle** (GPU, Internet on) with inputs `sherif31/group-activity-recognition-volleyball` and
`yoga0702/vball-orel` (private; research use). Outputs in `/kaggle/working/`: `runs/<mix>/best.onnx`, evaluation folders.

In [ ]:
import glob, os, shutil
WORK = '/tmp/work'; OUT = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)
VIDEOS = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/videos/47', recursive=True))
OREL = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/test/match1', recursive=True)[0]))  # holds train/ and test/
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q --depth 1 https://github.com/nttcom/WASB-SBDT {WORK}/wasb
!git clone -q --depth 1 https://github.com/DL-Volleyball-Analysis/volleyball-analysis {WORK}/repo
!git clone -q --depth 1 https://github.com/asigatchov/fast-volleyball-tracking-inference {WORK}/repo/external/fast-volleyball-tracking-inference
!mkdir -p {WORK}/repo/models && cp {WORK}/repo/external/fast-volleyball-tracking-inference/models/VballNetV4c_seq9_grayscale_*.onnx {WORK}/repo/models/vballnet_v4c.onnx
!pip -q install -e {WORK}/repo --no-deps
!pip -q install onnx onnxruntime-gpu gdown omegaconf
!gdown -q 1M9y4wPJqLc0K-z-Bo5DP8Ft5XwJuLqIS -O {WORK}/wasb_volleyball_best.pth.tar
!gdown -q 1urZpZiiepC85JD1u3VeURgUpztRgI0yl -O {WORK}/ball_ann.zip && unzip -q -o {WORK}/ball_ann.zip -d {WORK}/ann
ANN = f'{WORK}/ann/volleyball_ball_annotation'

In [ ]:
CLIPS = f'{WORK}/clips'
!cd {WORK}/repo && python scripts/build_ball_clips.py --out {CLIPS} --volleyball-videos {VIDEOS} --volleyball-ann {ANN} --orel {OREL}

In [ ]:
RUNS = f'{OUT}/runs'
for mix, args in (('gym', 'orel=1'), ('mixed', 'volleyball=0.5 orel=0.5')):
    !cd {WORK}/repo && python training/ball/train_wasb.py --clips {CLIPS} --wasb-src {WORK}/wasb/src --init {WORK}/wasb_volleyball_best.pth.tar --out {RUNS}/{mix} --mix {args} --epochs 8 --samples 24000 --batch 16 --eval-init

In [ ]:
# evaluate: WASB's published weights (exported by the same code) and both runs, on broadcast and gym test matches
import sys, torch
sys.path.insert(0, f'{WORK}/repo/training/ball')
from train_wasb import build_model, with_sigmoid
ref = with_sigmoid(build_model(f'{WORK}/wasb/src', f'{WORK}/wasb_volleyball_best.pth.tar')).eval()
torch.onnx.export(ref, torch.randn(1, 9, 288, 512), f'{RUNS}/wasb_published.onnx', input_names=['frames'],
                  output_names=['heatmaps'], opset_version=17, dynamic_axes={'frames': {0: 'batch'}, 'heatmaps': {0: 'batch'}}, dynamo=False)
for name, onnx in (('published', f'{RUNS}/wasb_published.onnx'), ('gym', f'{RUNS}/gym/best.onnx'), ('mixed', f'{RUNS}/mixed/best.onnx')):
    if not os.path.exists(onnx):
        print('missing', onnx); continue
    shutil.copy(onnx, f'{WORK}/repo/models/wasb_volleyball.onnx')
    !cd {WORK}/repo && python scripts/eval_ball_detection.py --videos {VIDEOS} --ann {ANN} --detectors wasb --out {OUT}/eval/{name}_broadcast
    !cd {WORK}/repo && python scripts/eval_ball_detection.py --dataset orel --orel {OREL} --detectors wasb --out {OUT}/eval/{name}_gym